In [ ]:
import rasterio
import numpy as np
import geopandas as gpd
from ultralytics import YOLO
from shapely.geometry import Polygon
from rasterio.windows import Window
import pandas as pd

#======================================= CONFIGURATION ==============================================
IMAGE_PATH = r"C:/Users/juanm/Documents/Proyectos_personales/Proyectos_Git/Proyecto_Copas/Prueba_Tiling_4/Prueba_tiling_4.tif"
MODEL_PATH = r"C:/Users/juanm/Documents/Proyectos_personales/Proyectos_Git/Proyecto_Copas/Modelos_IA/Modelo_Nano_3_960/weights/best.pt"

TILE_SIZE = 960       # Window size for processing
OVERLAP = 200         # Overlap between tiles in pixels
CONF_THRESHOLD = 0.25 # Model detection confidence threshold
MIN_DIAMETER = 3      # Minimum crown diameter to filter noise and low scrub

#======================================= TILE PROCESSING ============================================
def run_forest_inventory_tiling():
    """Performs tree crown segmentation using a tiling approach for large raster images."""
    print(f"Loading model: {MODEL_PATH}...")
    model = YOLO(MODEL_PATH)
    all_polygons = []

    with rasterio.open(IMAGE_PATH) as src:
        full_transform = src.transform
        crs = src.crs
        width, height = src.width, src.height
        res_x, res_y = src.res
        
        # Calculate total analyzed area in hectares
        total_area_ha = (width * height * abs(res_x * res_y)) / 10000

        print(f"--- IMAGE INFO ---")
        print(f"Dimensions: {width}x{height} px | GSD: {abs(res_x)*100:.2f} cm/px")
        print(f"Total Area: {total_area_ha:.4f} ha")
        print(f"Processing {TILE_SIZE}x{TILE_SIZE} windows...")

        # Tiling loop
        for y in range(0, height, TILE_SIZE - OVERLAP):
            for x in range(0, width, TILE_SIZE - OVERLAP):
                w, h = min(TILE_SIZE, width - x), min(TILE_SIZE, height - y)
                if w < 32 or h < 32: continue

                # Define window and read RGB bands (1, 2, 3)
                window = Window(x, y, w, h)
                tile_img = np.moveaxis(src.read([1, 2, 3], window=window), 0, -1)

                # Inference
                results = model.predict(tile_img, imgsz=TILE_SIZE, conf=CONF_THRESHOLD, device=0, verbose=False)

                if results[0].masks is not None:
                    for mask in results[0].masks.xy:
                        if len(mask) < 3: continue
                        
                        # Step 1: Tile pixel coordinates to Global Image pixel coordinates
                        global_pixel_coords = [(p[0] + x, p[1] + y) for p in mask] 
                        
                        # Step 2: Global Pixel to Real World Coordinates (CRS) using Affine Transformation
                        geo_coords = [full_transform * p for p in global_pixel_coords]
                        all_polygons.append(Polygon(geo_coords))

    if not all_polygons:
        print("No trees detected.")
        return

    #========================== DUPLICATE CLEANING AND TOPOLOGY ==========================
    print("Cleaning overlapping duplicates and repairing geometry...")
    gdf_raw = gpd.GeoDataFrame({'geometry': all_polygons}, crs=crs)
    
    # Fix invalid polygons (prevents GEOS TopologyException)
    gdf_raw["geometry"] = gdf_raw.geometry.buffer(0)
    gdf_raw = gdf_raw[gdf_raw.is_valid & ~gdf_raw.is_empty].copy()
    
    # Sort by area to prioritize larger, complete detections in NMS
    gdf_raw["temp_area"] = gdf_raw.geometry.area
    gdf_raw = gdf_raw.sort_values("temp_area", ascending=False)

    clean_trees = []
    used_indices = set()
    sindex = gdf_raw.sindex
    
    # Custom Non-Maximum Suppression (NMS) based on spatial intersection
    for i, row in gdf_raw.iterrows():
        if i in used_indices: continue
        
        possible_matches_idx = list(sindex.intersection(row.geometry.bounds))
        for j_pos in possible_matches_idx:
            j = gdf_raw.index[j_pos]
            if i == j or j in used_indices: continue
            
            match = gdf_raw.loc[j]
            try:
                # If overlap exceeds 50%, discard the smaller detection
                if row.geometry.intersects(match.geometry):
                    inter_area = row.geometry.intersection(match.geometry).area
                    if inter_area > (match.geometry.area * 0.5):
                        used_indices.add(j)
            except:
                used_indices.add(j) # Safety removal if topology check fails
        
        clean_trees.append(row.geometry)

    #======================================= FINAL RESULTS ================================================
    gdf = gpd.GeoDataFrame({'geometry': clean_trees}, crs=crs)
    gdf["Area_sqm"] = gdf.geometry.area.round(2)
    gdf["Crown_Diam_m"] = (2 * np.sqrt(gdf["Area_sqm"] / np.pi)).round(2)
    
    # Filter by minimum crown diameter
    gdf = gdf[gdf["Crown_Diam_m"] >= MIN_DIAMETER].copy()
    
    # Attribute table setup
    gdf["ID"] = range(1, len(gdf) + 1)
    centroids = gdf.geometry.centroid
    gdf["Coordinates"] = [f"{round(c.x, 2)} {round(c.y, 2)}" for c in centroids]
    
    # Forest Statistics
    diameters = gdf["Crown_Diam_m"]
    canopy_cover_pct = (gdf["Area_sqm"].sum() / (total_area_ha * 10000)) * 100

    print(f"\n" + "="*45)
    print(f"          FOREST INVENTORY STATISTICS")
    print(f"="*45)
    print(f"  > Total Area Analyzed:      {total_area_ha:.4f} ha")
    print(f"  > Detected Stems:           {len(gdf)} trees")
    print(f"  > Density:                  {len(gdf)/total_area_ha:.2f} stems/ha")
    print(f"  > Canopy Cover (CC):        {canopy_cover_pct:.2f} %")
    print(f"---" + "-"*42)
    print(f"  > Mean Crown Diameter:      {diameters.mean():.2f} m")
    print(f"  > Std Deviation (Diam):     {diameters.std():.2f} m")
    print(f"  > Min Diameter:             {diameters.min():.2f} m")
    print(f"  > Max Diameter:             {diameters.max():.2f} m")
    print(f"  > Median Diameter:          {diameters.median():.2f} m")
    print(f"="*45)

    # Export selected columns
    output_name = "Forest_Inventory_Results.gpkg"
    gdf[["ID", "Area_sqm", "Crown_Diam_m", "Coordinates", "geometry"]].to_file(output_name, driver='GPKG')
    print(f"\nFile generated: {output_name}")

if __name__ == "__main__":
    run_forest_inventory_tiling()

Cargando modelo: C:/Users/juanm/Documents/Proyectos_personales/Proyectos_Git/Proyecto_Copas/Modelos_IA/Modelo_Nano_3_960/weights/best.pt...
--- INFO IMAGEN ---
Dimensiones: 4443x2709 px | GSD: 25.00 cm/px
Superficie total: 75.2255 ha
Procesando ventanas de 960x960...
Limpiando duplicados de solape y corrigiendo geometría...

      ESTADÍSTICOS DEL INVENTARIO FORESTAL
  > Superficie analizada:      75.2255 ha
  > Árboles detectados:        3643 pies
  > Densidad:                  48.43 pies/ha
  > Fcc:                       18.08 %
---------------------------------------------
  > Diámetro medio:            6.68 m
  > Desviación típica diám.:   1.72 m
  > Diámetro mínimo:           3.05 m
  > Diámetro máximo:           13.44 m
  > Diámetro mediana:          6.51 m

 Archivo generado: Prueba_Tiling_4.gpkg
